In [2]:
import numpy as np 
import pandas as pd 

In [4]:
balls = pd.read_csv('../data/raw/ipl_ball_by_ball_2008_2026.csv', low_memory=False)
matches = pd.read_csv('../data/raw/ipl_matches_2008_2026.csv')
season_stats = pd.read_csv('../data/raw/ipl_player_season_stats_2008_2026.csv')

In [5]:
balls.head()

,match_id,season,date,venue,innings,is_super_over,batting_team,bowling_team,over,ball,...,team_score,team_wickets,legal_balls_bowled,current_run_rate,target,runs_needed,balls_left,required_run_rate,review_by,review_decision
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,1,...,1,0,1,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,2,...,1,0,2,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,3,...,2,0,2,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,3,...,2,0,3,4.0,NaN,NaN,NaN,NaN,NaN,NaN
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,False,Kolkata Knight Riders,Royal Challengers Bengaluru,1,4,...,2,0,4,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
balls.columns

Index(['match_id', 'season', 'date', 'venue', 'innings', 'is_super_over',
       'batting_team', 'bowling_team', 'over', 'ball', 'delivery_in_over',
       'phase', 'batter', 'non_striker', 'bowler', 'batter_runs', 'extra_runs',
       'total_runs', 'extra_type', 'is_legal_ball', 'is_four', 'is_six',
       'is_dot', 'is_wicket', 'player_out', 'dismissal_kind', 'fielders',
       'bowler_wicket', 'team_score', 'team_wickets', 'legal_balls_bowled',
       'current_run_rate', 'target', 'runs_needed', 'balls_left',
       'required_run_rate', 'review_by', 'review_decision'],
      dtype='object')

In [7]:
balls['batter'].value_counts()

batter
V Kohli           7127
RG Sharma         5659
S Dhawan          5483
DA Warner         4849
AM Rahane         4391
                  ... 
NJ Rimmington        1
S Kaushik            1
PVSN Raju            1
V Pratap Singh       1
Sunny Gupta          1
Name: count, Length: 738, dtype: int64

In [12]:
balls.groupby('batter')['batter_runs'].sum().sort_values(ascending=False).sample(10)

batter
SB Joshi                6
RA Jadeja            3526
S Badrinath          1441
TM Srivastava           8
P Parameswaran          1
Washington Sundar     888
HC Brook              190
TK Curran             127
WG Jacks              602
Anureet Singh          36
Name: batter_runs, dtype: int64

In [14]:
from pathlib import Path

# Columns that can contain player names
player_columns = [
    col for col in [
        "batter",
        "non_striker",
        "bowler",
        "player_out",
        "fielder",
        "fielders"
    ]
    if col in balls.columns
]

if not player_columns:
    raise ValueError("No player columns found in the balls DataFrame.")

# Combine all player names
players = pd.concat(
    [balls[col] for col in player_columns],
    ignore_index=True
)

# Handle columns containing multiple fielders
players = (
    players.dropna()
    .astype(str)
    .str.split(r"[,;|]")
    .explode()
    .str.strip()
)

# Remove empty and invalid values
players = players[
    ~players.str.lower().isin(["", "nan", "none", "null", "-"])
]

# Remove duplicates and sort
players = sorted(players.unique(), key=str.lower)

# Save CSV
output_path = Path("../data/processed/ipl_players.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

player_df = pd.DataFrame({
    "player_name": players
})

player_df.to_csv(output_path, index=False)

print(f"Total unique players: {len(player_df)}")
print(f"Saved successfully: {output_path}")

Total unique players: 816
Saved successfully: ..\data\processed\ipl_players.csv
